# SNMF TCGA - gene level analysis
* SNMF model trained on Gene KO cell line data
* Evaluate generalizability w.r.t. TCGA data

In [ ]:
# snmf_env
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm
from matplotlib import cm
import os
import pickle as pkl
import umap

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn import metrics

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

from SigProfilerAssignment import Analyzer as Analyze

%matplotlib inline
sns.set_theme(style="whitegrid")
sns.set(rc={'figure.figsize':(10.0,8.27)})

from SigProfilerMatrixGenerator import install as genInstall
genInstall.install('GRCh37')


# LOAD Bootstrapped functions
import sys
import os
# append project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

import importlib
from src.processing import bootstrapping as boot
# Reload the module to ensure we have the latest version
importlib.reload(boot)


# test SNMF on TCGA data

In [ ]:
# load TCGA data
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

cancer = 'BRCA'  # Example cancer type, change as needed
TCGA_count = pd.read_pickle(os.path.join(project_root, 'data', 'processed', 'TCGA','profiles','count', '{}.pkl'.format(cancer)))
TCGA_profiles = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'TCGA','profiles','norm_SBS', '{}.text'.format(cancer)), sep = '\t', index_col=0)

print(TCGA_count.iloc[:10, ]['count_SBS'])
print(TCGA_count.shape)

TCGA_profiles

## Run SNMF test on TCGA

In [ ]:
import os
import pandas as pd
import importlib
from SNMF import sigpro as sig
importlib.reload(sig)

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

# SNMF Parameters
min_k = max_k = 5
reps = 10
lr = 5e-3
l_p = 0.0001
cancer_list = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']

bootstraps = ["multinomial", "dirichlet"]
lc_path_map = {0: 0.0, 1: 0.1}  # folder suffix → actual lambda_c value

results_df = pd.DataFrame()

for bootstrap in bootstraps:
    for lc_suffix, lc_value in lc_path_map.items():
        print(f"\n=== Running {bootstrap}_lc{lc_suffix} (λ_c={lc_value}) ===")
        
        model_root = os.path.join( project_root, "results", "analysis", "bootstrap_comparison_new", f"SNMF_results_{bootstrap}_lc{lc_suffix}", "run_10")
        output_root = os.path.join(project_root, "results", "reproduce", "TCGA", f"{bootstrap}_lc{lc_suffix}")
        
        for cancer in cancer_list:
            print(f"  → {cancer}")
            X_tcga = os.path.join(project_root, 'data', 'processed', 'TCGA', 'profiles', 'norm_SBS', f"{cancer}.text")
            Y_tcga = os.path.join(project_root, 'data', 'processed', 'TCGA', 'annotations', cancer, "Y_cancer.text")
            output_path = os.path.join(output_root, cancer, "SNMF_results")
            os.makedirs(output_path, exist_ok=True)

            # Run SNMF Test
            acc, f1, rec = sig.test_sigProfilerExtractor(
                input_type="text",
                model_path=model_root,
                output=output_path,
                test_data=X_tcga,
                test_label=Y_tcga,
                minimum_signatures=min_k,
                maximum_signatures=max_k,
                nmf_replicates=reps,
                lambda_c=lc_value,
                lambda_p=l_p,
                lr=lr,
                filter=False,
                make_decomposition_plots=False
            )

            results_df = pd.concat([
                results_df,
                pd.DataFrame([{
                    'bootstrap': bootstrap, 'lambda_c': lc_value,
                    'cancer': cancer, 'k': min_k, 'lambda_p': l_p, 'lr': lr,
                    'acc_test': acc, 'f1_test': f1, 'rec_test': rec
                }])
            ], ignore_index=True)

# Save results
results_path = os.path.join(output_root, "SNMF_TCGA_test_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved results to {results_path}")


In [ ]:
from SNMF import sigpro as sig
importlib.reload(sig)

# --- Configuration ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
#! REPLACE with reproduction path -> final SNMF
model_root = os.path.join(project_root, "results", "analysis", "bootstrap_comparison_new", "SNMF_results_multinomial")
output_root = os.path.join(project_root, "results", "reproduce", "TCGA")

# SNMF Parameters
min_k = max_k = 5
reps = 10
lr = 5e-3
l_c = 0.1
l_p = 0.0001

# Cancer types
cancer_list = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']
results_df = pd.DataFrame()

for cancer in cancer_list:
    print(f"Running SNMF test for {cancer}...")

    # --- Paths ---
    X_tcga = os.path.join(project_root, 'data', 'processed', 'TCGA', 'profiles', 'norm_SBS', f"{cancer}.text")
    Y_tcga = os.path.join(project_root, 'data', 'processed', 'TCGA', 'annotations', cancer, "Y_cancer.text")
    model_path = os.path.join(model_root)
    output_path = os.path.join(output_root, cancer, "SNMF_results")
    os.makedirs(output_path, exist_ok=True)

    # --- Run SNMF Test ---
    acc, f1, rec = sig.test_sigProfilerExtractor(
        input_type="text",
        model_path=model_path,
        output=output_path,
        test_data=X_tcga,
        test_label=Y_tcga,
        minimum_signatures=min_k,
        maximum_signatures=max_k,
        nmf_replicates=reps,
        lambda_c=l_c,
        lambda_p=l_p,
        lr=lr,
        filter=False,
        make_decomposition_plots=False
    )

    # --- Store results ---
    results_df = pd.concat([
        results_df,
        pd.DataFrame([{
            'cancer': cancer, 'k': min_k, 'lambda_c': l_c, 'lambda_p': l_p, 'lr': lr,
            'acc_test': acc, 'f1_test': f1, 'rec_test': rec
        }])
    ], ignore_index=True)

# --- Save All Results ---
results_path = os.path.join(output_root, "SNMF_TCGA_test_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved results to {results_path}")


## Run SigProfiler - COSMIC

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

cancer_list = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']

for cancer in cancer_list:
    print(f"\nProcessing {cancer}...")

    # Paths for normalized SBS profiles
    X_tcga_norm_path = os.path.join(project_root, "data", "processed", "TCGA", "profiles", "norm_SBS", f"{cancer}.text")
    X_tcga_norm_scaled_path = os.path.join(project_root, "data", "processed", "TCGA", "profiles", "norm_SBS", f"{cancer}_1000x.text")

    # Load norm SBS, multiply by 1000, save for SigProfiler
    df_norm = pd.read_csv(X_tcga_norm_path, sep="\t", index_col=0)
    df_norm_scaled = df_norm * 1000
    df_norm_scaled.to_csv(X_tcga_norm_scaled_path, sep="\t", index_label="MutationType")

    print(f" - Normalized SBS profiles scaled and saved to {X_tcga_norm_scaled_path}")

    # Paths for counts
    counts_pkl_path = os.path.join(project_root, "data", "processed", "TCGA", "profiles", "count", f"{cancer}.pkl")
    counts_text_path = os.path.join(project_root, "data", "processed", "TCGA", "profiles", "count", f"{cancer}.text")

    # Load counts pkl, transpose, keep first 96 rows, save as text
    df_counts = pd.read_pickle(counts_pkl_path)
    df_counts = df_counts.T.iloc[:96, :]
    df_counts.to_csv(counts_text_path, sep="\t", index_label="MutationType")

    print(f" - Count matrix transposed, trimmed to 96 SBS, and saved as text to {counts_text_path}")

    # Run COSMIC fit on normalized SBS and counts
    output_dir_norm = os.path.join(project_root, "results", "fit_cosmic", f"{cancer}_norm")
    output_dir_counts = os.path.join(project_root, "results", "fit_cosmic", f"{cancer}_counts")
    os.makedirs(output_dir_norm, exist_ok=True)
    os.makedirs(output_dir_counts, exist_ok=True)

    print(f" - Running COSMIC fit on counts matrix...")
    Analyze.cosmic_fit(
        samples=counts_text_path,
        output=output_dir_counts,
        input_type="matrix",
        context_type="96",
        verbose=True
    )

    print(f" - Running COSMIC fit on normalized SBS profiles...")
    Analyze.cosmic_fit(
        samples=X_tcga_norm_scaled_path,
        output=output_dir_norm,
        input_type="matrix",
        context_type="96",
        verbose=True
    )


print("\nAll COSMIC fits done!")


# Load data (adata)

In [ ]:
import sys
import os
os.getcwd()
sys.path.append('src/reproduce')  # add folder to path if needed
os.path.exists("src/reproduce/utils_tcga.py")

# import importlib
# import src.reproduce.utils_tcga as utils_tcga
# importlib.reload(utils_tcga)

%load_ext autoreload
%autoreload 2

from src.reproduce.utils_tcga import (
    build_tcga_adata,
    build_tcga_multicancer_adata,
    run_mannwhitney_gene_signature_test,
    plot_ddr_signature, 
    get_significant_genes_from_adata_df,
    plot_mw_gene_stats
)


In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
pathway_gene = pd.read_csv(os.path.join(project_root, "data", "processed", "TCGA", "pathway_gene_matrix.csv"), index_col=0)

adata_hrd= build_tcga_adata('BRCA', project_root, bootstrap='multinomial')
# adata_brca.write(os.path.join(output_dir_base, "BRCA_tcga_annotated.h5ad"))

mmrd_cancers = ['COAD', 'UCEC', 'STAD']
adata_mmrd = build_tcga_multicancer_adata(mmrd_cancers, project_root, bootstrap='multinomial')
adata_mmrd

# Gene level analysis
* HRd in BRCA
* MMRd in COAD/UCEC/STAD

## Analysis Paper

In [ ]:

adata_hrd = run_mannwhitney_gene_signature_test(
    adata=adata_hrd,
    sig_matrix_key='exposures_test',
    sig_col='SBS96A_HR',
    verbose=True,
    plot=True,
    ddr='HR',
    fdr_thresh=0.01,
    project_root=project_root,         
    save_dir='results/figures/sup/tcga' 
)

adata_mmrd = run_mannwhitney_gene_signature_test(
    adata=adata_mmrd,
    sig_matrix_key='exposures_test',
    sig_col='SBS96B_MMR',
    verbose=True,
    plot=True,
    ddr='MMR',
    fdr_thresh=0.05,
    project_root=project_root,          
    save_dir='results/figures/sup/tcga'
)


In [ ]:
# ----- HR -----
sig_genes_hr = get_significant_genes_from_adata_df(
    adata=adata_hrd,
    mw_results_key="MW_results_SBS96A_HR",
    fdr_thresh=0.01
)
print("HR significant genes:", sig_genes_hr)

plot_ddr_signature(
    adata_hrd,
    sig_genes_hr,
    ddr_label="HR",
    project_root=project_root,
    save_dir="results/figures/sup/tcga",
    save_basename=None  # or set a custom name
)

# ----- MMR -----
sig_genes_mmr = get_significant_genes_from_adata_df(
    adata=adata_mmrd,
    mw_results_key="MW_results_SBS96B_MMR",
    fdr_thresh=0.05
)
print("MMR significant genes:", sig_genes_mmr)

plot_ddr_signature(
    adata_mmrd,
    sig_genes_mmr,
    ddr_label="MMR",
    project_root=project_root,
    save_dir="results/figures/sup/tcga",
    save_basename=None
)


In [ ]:
genes = plot_mw_gene_stats(
    adata=adata_hrd,
    mw_sig="SBS96A_HR",
    annotate=True,
    pval_threshold=0.01,
    verbose=True,
    project_root=project_root 
)

genes = plot_mw_gene_stats(
    adata=adata_mmrd,
    mw_sig="SBS96B_MMR",
    annotate=True,
    pval_threshold=0.05,
    verbose=True,
    project_root=project_root
)


In [ ]:
# --- Config ---
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
save_dir = "results/figures/sup/tcga"

# Map signature → DDR label used by the plotting/saving functions
sig_to_ddr = {
    "SBS96A_HR":        "HR",
    "SBS96B_MMR":       "MMR",
    "SBS96C_Control":   "Control",
    "SBS96D_BER_UNG":   "BER",
    "SBS96E_BER_OGG1":  "BER",
}

# Choose which AnnData to use per DDR (HR/MMR you already split; others pick what’s appropriate)
ddr_to_adata = {
    "HR":      adata_hrd,
    "MMR":     adata_mmrd,
    "BER":     adata_mmrd,   # change if you prefer adata_hrd
    "Control": adata_hrd,    # or adata_mmrd; your call
}

sig_list = ["SBS96A_HR", "SBS96B_MMR", "SBS96C_Control", "SBS96D_BER_UNG", "SBS96E_BER_OGG1"]

for sig_col in sig_list:
    ddr = sig_to_ddr[sig_col]
    adata_in = ddr_to_adata[ddr]

    # 1) Mann–Whitney tests (+ boxplots saved under sup/tcga/<DDR>/...)
    adata_out = run_mannwhitney_gene_signature_test(
        adata=adata_in,
        sig_matrix_key="exposures_test",
        sig_col=sig_col,
        verbose=True,
        plot=True,
        ddr=ddr,
        fdr_thresh=0.01 if ddr=="HR" else 0.05,   # your earlier thresholds
        project_root=project_root,
        save_dir=save_dir
    )

    # 2) Extract significant genes from the stored MW results
    mw_key = f"MW_results_{sig_col}"
    significant_genes = get_significant_genes_from_adata_df(
        adata=adata_out,
        mw_results_key=mw_key,
        fdr_thresh=0.01 if ddr=="HR" else 0.05
    )
    print(f"[{sig_col}] Significant genes:", significant_genes)

    if len(significant_genes) == 0:
        print(f"  → No significant genes found for {sig_col}, skipping plotting.")
        continue
    # 3) Plot DDR signature heat figure (saved as PDF only)
    plot_ddr_signature(
        adata_out,
        significant_genes,
        ddr_label=ddr,
        project_root=project_root,
        save_dir=save_dir,
        save_basename=None  # auto filename
    )

    # 4) MW stats scatter (saved as PDF only)
    plot_mw_gene_stats(
        adata=adata_out,
        mw_sig=sig_col,
        annotate=True,
        pval_threshold=0.01 if ddr=="HR" else 0.05,
        verbose=True,
        project_root=project_root,
        save_dir=save_dir
    )


## On Mutation count (TMB) instead of exposures 

In [ ]:
# ----- Mutation Count -----
adata_mutcount = run_mannwhitney_gene_signature_test(
    adata=adata_hrd,
    sig_matrix_key="mutation_count",
    sig_col=0,               # first column
    verbose=True,
    plot=True,
    ddr="HR",                # folder under sup/tcga/HR
    fdr_thresh=0.01,
    project_root=project_root,
    save_dir="results/figures/sup/tcga"
)

# Extract significant genes
significant_genes_mut = get_significant_genes_from_adata_df(
    adata=adata_mutcount,
    mw_results_key="MW_results_0",  # because sig_col=0
    fdr_thresh=0.01
)
print("Significant genes for mutation count:", significant_genes_mut)

# Plot MW stats scatter (will save as PDF)
genes_mut = plot_mw_gene_stats(
    adata=adata_mutcount,
    mw_sig="0",   # <- better basename than 0
    annotate=True,
    pval_threshold=0.01,
    verbose=True,
    project_root=project_root,
    save_dir="results/figures/sup/tcga"
)


## Gene correlated with COSMIC signatures 
Verification:
Sum DDRd related signatures in COSMIC (for HRd and MMRd) > similar genes show up?

In [ ]:
mmr_cosmic_signatures = ['SBS6', 'SBS14', 'SBS15', 'SBS20', 'SBS21', 'SBS26', 'SBS44']

# Sum the MMR-related cosmic exposures
cosmic_exp_df = pd.DataFrame(adata_mmrd.uns['cosmic_exposures_norm'], index=adata_mmrd.obs_names)

# Make sure all signatures exist
missing_sigs = [sig for sig in mmr_cosmic_signatures if sig not in cosmic_exp_df.columns]
if missing_sigs:
    print(f"Warning: These signatures are missing from cosmic_exposures_norm: {missing_sigs}")

# Compute summed MMR exposure (normalize if needed)
existing_sigs = [sig for sig in mmr_cosmic_signatures if sig in cosmic_exp_df.columns]
cosmic_mmr_sum = cosmic_exp_df[existing_sigs].sum(axis=1) / 1000

# Store as DataFrame with explicit column name
adata_mmrd.uns['cosmic_mmr'] = pd.DataFrame(
    {"cosmic_MMR_sum": cosmic_mmr_sum}, index=adata_mmrd.obs_names
)

# --- Run MW test on summed COSMIC MMR exposure ---
adata_temp = run_mannwhitney_gene_signature_test(
    adata=adata_mmrd,
    sig_matrix_key='cosmic_mmr',
    sig_col="cosmic_MMR_sum",   # <- use column name instead of 0
    verbose=False,
    plot=True,
    ddr="MMR",
    fdr_thresh=0.01,
    save_dir="results/figures/sup/tcga"
)

# --- MW stats scatterplot ---
genes_mut = plot_mw_gene_stats(
    adata=adata_temp,
    mw_sig="cosmic_MMR_sum",    # <- matches column name above
    annotate=True,
    pval_threshold=0.01,
    verbose=True,
    save_dir="results/figures/sup/tcga"
)




In [ ]:
# --- HR COSMIC sum from adata_hrd ---
hr_cosmic_signatures = ['SBS3', 'SBS8']

# Get COSMIC exposures as DataFrame
cosmic_exp_df = pd.DataFrame(adata_hrd.uns['cosmic_exposures_norm'], index=adata_hrd.obs_names)

# Check presence
missing_sigs = [sig for sig in hr_cosmic_signatures if sig not in cosmic_exp_df.columns]
if missing_sigs:
    print(f"Warning: missing COSMIC signatures for HR: {missing_sigs}")

# Sum existing HR signatures (optionally scaled)
existing_sigs = [sig for sig in hr_cosmic_signatures if sig in cosmic_exp_df.columns]
cosmic_hr_sum = cosmic_exp_df[existing_sigs].sum(axis=1) / 1000.0

# Store as single-column DataFrame with clear name
adata_hrd.uns['cosmic_hr'] = pd.DataFrame(
    {'cosmic_HR_sum': cosmic_hr_sum}, index=adata_hrd.obs_names
)

# --- MW test on summed HR COSMIC exposure ---
adata_temp = run_mannwhitney_gene_signature_test(
    adata=adata_hrd,
    sig_matrix_key='cosmic_hr',          # <- use the key we just stored
    sig_col='cosmic_HR_sum',             # <- column name, not 0
    verbose=False,
    plot=True,
    ddr='HR',                            # <- HR (not MMR)
    fdr_thresh=0.01,
    project_root=project_root,
    save_dir='results/figures/sup/tcga'
)

# --- MW stats scatter (PDF saved via plot_mw_gene_stats) ---
genes_mut = plot_mw_gene_stats(
    adata=adata_temp,
    mw_sig='cosmic_HR_sum',              # match the column name for clean filenames
    annotate=True,
    pval_threshold=0.01,
    verbose=True,
    project_root=project_root,
    save_dir='results/figures/sup/tcga'
)


### COSMIC HRd signatures low exposed?
* SBS3, SBS8 have low exposure in TCGA BRCA samples
* therefore, genes correlated with these signatures are not significant
* rerun SigProfiler cosmic_fit() with tuned parameters to get more exposure for these signatures?
    * Exome data
    * Exclude Artefact and 'Other' signatures

In [ ]:
import os
import pandas as pd
from SigProfilerAssignment import Analyzer as Analyze

# ---------------------------------------------------------------------
# 1. Define project directories
# ---------------------------------------------------------------------
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
cancer = "BRCA"

print(f"\n=== Running updated COSMIC fit for {cancer} ===")

# Input and output paths
counts_text_path = os.path.join(
    project_root, "data", "processed", "TCGA", "profiles", "count", f"{cancer}.text"
)
output_dir_counts_exome = os.path.join(
    project_root, "results", "fit_cosmic", f"{cancer}_counts_exome_exclude"
)
os.makedirs(output_dir_counts_exome, exist_ok=True)

# ---------------------------------------------------------------------
# 2. Run COSMIC fit (using exome normalization, counts input)
# ---------------------------------------------------------------------
print(" - Running COSMIC fit (exome=True, counts input)...")
Analyze.cosmic_fit(
    samples=counts_text_path,
    output=output_dir_counts_exome,
    input_type="matrix",
    context_type="96",
    cosmic_version=3.4,
    exome=True,  # <-- IMPORTANT FIX
    genome_build="GRCh37",
    exclude_signature_subgroups=["Artefact", "Other"],
    make_plots=True,
    verbose=True
)

print(" - COSMIC fitting complete!\n")


In [ ]:
# --- HR COSMIC sum from adata_hrd ---
hr_cosmic_signatures = ['SBS3', 'SBS8']

# Get COSMIC exposures as DataFrame
# cosmic_exp_df = pd.DataFrame(adata_hrd.uns['cosmic_exposures_norm'], index=adata_hrd.obs_names)
cosmic_exp_df = pd.DataFrame(adata_hrd.uns['cosmic_exposures_counts_exome_exclude'], index=adata_hrd.obs_names)


# cosmic_exposures_counts_exome

# Check presence
missing_sigs = [sig for sig in hr_cosmic_signatures if sig not in cosmic_exp_df.columns]
if missing_sigs:
    print(f"Warning: missing COSMIC signatures for HR: {missing_sigs}")

# Sum existing HR signatures (optionally scaled)
existing_sigs = [sig for sig in hr_cosmic_signatures if sig in cosmic_exp_df.columns]
cosmic_hr_sum = cosmic_exp_df[existing_sigs].sum(axis=1) / 1000.0

# Store as single-column DataFrame with clear name
adata_hrd.uns['cosmic_hr'] = pd.DataFrame(
    {'cosmic_HR_sum': cosmic_hr_sum}, index=adata_hrd.obs_names
)

# --- MW test on summed HR COSMIC exposure ---
adata_temp = run_mannwhitney_gene_signature_test(
    adata=adata_hrd,
    sig_matrix_key='cosmic_hr',          # <- use the key we just stored
    sig_col='cosmic_HR_sum',             # <- column name, not 0
    verbose=False,
    plot=True,
    ddr='HR',                            # <- HR (not MMR)
    fdr_thresh=0.01,
    project_root=project_root,
    save_dir='results/figures/sup/tcga'
)

# --- MW stats scatter (PDF saved via plot_mw_gene_stats) ---
genes_mut = plot_mw_gene_stats(
    adata=adata_temp,
    mw_sig='cosmic_HR_sum',              # match the column name for clean filenames
    annotate=True,
    pval_threshold=0.01,
    verbose=True,
    project_root=project_root,
    save_dir='results/figures/sup/tcga'
)


# Supplementary Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr

# --- Combine HRD + MMRD ---
df = pd.DataFrame({
    'mutation_count': pd.concat([adata_hrd.uns['mutation_count'], adata_mmrd.uns['mutation_count']]),
    'het': pd.concat([adata_hrd.uns['gene_het'].sum(1), adata_mmrd.uns['gene_het'].sum(1)]),
    'hom': pd.concat([adata_hrd.uns['gene_hom'].sum(1), adata_mmrd.uns['gene_hom'].sum(1)]),
    'cancer_type': ['BRCA'] * adata_hrd.n_obs + adata_mmrd.obs['cancer_type'].tolist()
})

# --- Filter ---
# df = df[df['mutation_count'] < 500]

# --- Plot ---
fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

for col, ax, title in zip(['het', 'hom'], axes, ['Heterozygous', 'Homozygous']):
    legend_labels = []
    for ct, g in df.groupby('cancer_type'):
        ax.scatter(g['mutation_count'], g[col], alpha=0.6)
        # Pearson correlation
        if len(g) > 1:
            r, p = pearsonr(g['mutation_count'], g[col])
            legend_labels.append(f"{ct} (r={r:.2f}, p={p:.2e})")
        else:
            legend_labels.append(f"{ct} (r=NA, p=NA)")
    
    ax.set_title(title)
    ax.set_ylabel('Mutated Genes')
    ax.grid(True)
    ax.set_xscale('log')  # log scale for mutation counts
    ax.legend(legend_labels)

axes[-1].set_xlabel('Total Mutation Count')

plt.tight_layout()

# --- Save to sup/tcga ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
save_path = os.path.join(fig_dir, "mutation_count_vs_het_hom_logscale.pdf")
plt.savefig(save_path, bbox_inches="tight")
print(f"Saved plot to {save_path}")

plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr

# --- Combine HRD + MMRD ---
df = pd.DataFrame({
    "mutation_count": pd.concat([adata_hrd.uns["mutation_count"], adata_mmrd.uns["mutation_count"]]),
    "het": pd.concat([adata_hrd.uns["gene_het"].sum(1), adata_mmrd.uns["gene_het"].sum(1)]),
    "hom": pd.concat([adata_hrd.uns["gene_hom"].sum(1), adata_mmrd.uns["gene_hom"].sum(1)]),
    "cancer_type": ["BRCA"] * adata_hrd.n_obs + adata_mmrd.obs["cancer_type"].tolist()
})

# --- Plot side by side ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for col, ax, title in zip(["het", "hom"], axes, ["Heterozygous", "Homozygous"]):
    legend_labels = []
    for ct, g in df.groupby("cancer_type"):
        ax.scatter(g["mutation_count"], g[col], alpha=0.6)
        # Pearson correlation
        if len(g) > 1:
            r, p = pearsonr(g["mutation_count"], g[col])
            legend_labels.append(f"{ct} (r={r:.2f}, p={p:.2e})")
        else:
            legend_labels.append(f"{ct} (r=NA, p=NA)")

    ax.set_title(title)
    ax.set_xlabel("Total Mutation Count")
    ax.grid(True, color="lightgrey", linestyle="--", linewidth=0.5)
    ax.legend(legend_labels, fontsize=9)
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1)

axes[0].set_ylabel("Mutated Genes")
plt.tight_layout()

# --- Save to sup/tcga ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
save_path = os.path.join(fig_dir, "mutation_count_vs_het_hom_side_by_side.pdf")
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"✅ Saved plot to {save_path}")

plt.show()


## Detect associated genes 
Mann–Whitney U tests to detect genes whose mutated status is associated with changes in molecular signatures.

* For each gene in all_genes, split samples into:
    * WT (wild-type)
    * HET/HOM (mutated)
* Compare the distribution of some marker (e.g., SBS3 exposure) between these two groups.
* Use the Mann–Whitney U test (non-parametric) to check whether mutants show significantly higher values.
* Correct p-values (FDR).
* Identify significant genes and visualize them using boxplots.

## CHECK COSMIC associated genes 
per COSMIC signature instead of sum per DDRd.
Result, noisy:
- some (highly similar) COSMIC signature are almost mutually exclusive (e.g., SBS6 and SBS15 for MMRd) 
- therefore, summing them up is better.

In [ ]:
import os

# --- Config ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
save_dir = "results/figures/sup/tcga"

# MMR-related COSMIC signatures
mmr_cosmic_signatures = ['SBS6', 'SBS14', 'SBS15', 'SBS20', 'SBS21', 'SBS26', 'SBS44']

# Get available COSMIC columns directly from the matrix
cosmic_mat = pd.DataFrame(adata_mmrd.uns['cosmic_exposures_norm'], index=adata_mmrd.obs_names)
cosmic_cols = set(cosmic_mat.columns)

for sig in mmr_cosmic_signatures:
    if sig in cosmic_cols:
        print(f"\n=== Running Mann-Whitney test for COSMIC {sig} (MMR) ===")

        # 1) MW test on the COSMIC matrix column 'sig'
        adata_mmrd = run_mannwhitney_gene_signature_test(
            adata=adata_mmrd,
            sig_matrix_key='cosmic_exposures_norm',  # <- matrix key
            sig_col=sig,                              # <- column name
            ddr='MMR',
            verbose=True,
            plot=False,                               # plot after filtering
            fdr_thresh=0.01,
            project_root=project_root,
            save_dir=save_dir
        )

        mw_key = f"MW_results_{sig}"
        if mw_key not in adata_mmrd.uns:
            print(f"Warning: MW results key '{mw_key}' not found. Skipping {sig}.")
            continue

        # 2) Get significant genes (sorted by ascending FDR)
        significant_genes = get_significant_genes_from_adata_df(
            adata=adata_mmrd,
            mw_results_key=mw_key,
            fdr_thresh=0.01
        )

        # Optionally add TP53 if not already included
        if "TP53" not in significant_genes:
            significant_genes.append("TP53")

        print(f"Significant genes for {sig}: {significant_genes}")

        # 3) Plot DDR signature (PDF saved under sup/tcga/MMR/)
        plot_ddr_signature(
            adata=adata_mmrd,
            genes=significant_genes,
            ddr_label='MMR',
            project_root=project_root,
            save_dir=save_dir,
            save_basename=f"MMR_{sig}_EYcosmic_mutMatrix_{len(significant_genes)}genes"
        )

        # 4) (Optional) MW stats scatter for this signature (PDF saved)
        plot_mw_gene_stats(
            adata=adata_mmrd,
            mw_sig=sig,
            pval_threshold=0.01,
            annotate=True,
            verbose=True,
            project_root=project_root,
            save_dir=save_dir
        )

    else:
        print(f"Signature {sig} not found in cosmic_exposures_norm.")
    


# Performance COSMIC Classifier (LR)

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn import metrics

# Parameters & settings
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
param_grid = {'C': np.logspace(-2, 2, 11)}
cancer_list = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']
settings = [("multinomial", "M"), ("dirichlet", "D")]

# Loop over bootstrap types
for bootstrap, prefix in settings:
    print(f"\n=== {bootstrap.upper()} ===")

    # Load exposures & labels
    X_train = pd.read_csv(os.path.join(project_root, "results", "analysis", "bootstrap_comparison", f"cosmic_fit_{bootstrap}", "train_all", "Assignment_Solution", "Activities", "Assignment_Solution_Activities.txt"), sep="\t", index_col=0)
    X_test = pd.read_csv(os.path.join(project_root, "results", "analysis", "bootstrap_comparison", f"cosmic_fit_{bootstrap}", "test", "Assignment_Solution", "Activities", "Assignment_Solution_Activities.txt"), sep="\t", index_col=0)
    y_train_df = pd.read_csv(os.path.join(project_root, "data", "processed", f"bootstrapped_{bootstrap}", "N_100", f"Yboot{prefix}_train_all.text"), sep="\t", index_col=0).T
    y_test_df = pd.read_csv(os.path.join(project_root, "data", "processed", f"bootstrapped_{bootstrap}", "N_1000", f"Yboot{prefix}_test.text"), sep="\t", index_col=0).T
    y_train, y_test = y_train_df.idxmax(1), y_test_df.idxmax(1)
    class_order = list(y_test_df.columns)

    # Build custom CV splits
    folds = {f"train{i}": [s for s in X_train.index if s.endswith(f"_train{i}")] for i in range(1, 4)}
    name_to_idx = {n: i for i, n in enumerate(X_train.index)}
    train_idx = [[name_to_idx[s] for s in folds[f"train{i}"] + folds[f"train{j}"]] for i, j in [(1, 2), (1, 3), (2, 3)]]
    val_idx = [[name_to_idx[s] for s in folds[f"train{k}"]] for k in [3, 2, 1]]
    cv_splits = list(zip(train_idx, val_idx))

    # Train logistic regression with grid search
    clf = GridSearchCV(LogisticRegression(solver="saga", multi_class="multinomial", penalty="l2", max_iter=1000, random_state=42),
                       param_grid, cv=cv_splits, scoring="f1_macro", refit=True)
    clf.fit(X_train, y_train.loc[X_train.index])
    y_pred = clf.predict(X_test)
    print(f"F1={metrics.f1_score(y_test, y_pred, average='macro'):.4f}, Acc={metrics.accuracy_score(y_test, y_pred):.4f}, Recall={metrics.recall_score(y_test, y_pred, average='macro'):.4f}, C={clf.best_params_['C']}")

    # Save trained model
    out_dir = os.path.join(project_root, "results", "analysis", "bootstrap_comparison", f"LR_cosmic_{bootstrap}")
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(clf.best_estimator_, os.path.join(out_dir, "logreg_cosmic.pkl"))

    # Predict for each TCGA cancer
    for cancer in cancer_list:
        c_file = os.path.join(project_root, "results", "fit_cosmic", f"{cancer}_norm", "Assignment_Solution", "Activities", "Assignment_Solution_Activities.txt")
        if not os.path.exists(c_file): continue
        exp = pd.read_csv(c_file, sep="\t", index_col=0)
        exp.index = ['-'.join(s.split('-')[:3]) for s in exp.index]  # Standardize sample IDs
        probs = clf.best_estimator_.predict_proba(exp)
        prob_df = pd.DataFrame(probs, columns=clf.best_estimator_.classes_, index=exp.index)[class_order]
        prob_df["predicted_class"] = prob_df.idxmax(1)
        prob_df.to_csv(os.path.join(out_dir, f"{cancer}_predictions.csv"))


In [ ]:
from sklearn.metrics import recall_score

# --- Config ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
settings = [
    ("multinomial", 0),
    ("multinomial", 1),
    ("dirichlet", 0),
    ("dirichlet", 1)
]
datasets = {
    "HRD": lambda bootstrap, lc: build_tcga_adata(
        "BRCA", project_root, bootstrap=bootstrap, lc_suffix=lc
    ),
    "MMRD": lambda bootstrap, lc: build_tcga_multicancer_adata(
        ["COAD", "UCEC", "STAD"], project_root, bootstrap=bootstrap, lc_suffix=lc
    )
}
pathways_of_interest = ["HR", "MMR", "BER"]

# --- Load mapping ---
pathway_gene = pd.read_csv(
    os.path.join(project_root, "data", "processed", "TCGA", "pathway_gene_matrix.csv"),
    index_col=0
)
pathway_to_genes = {
    p: pathway_gene.columns[pathway_gene.loc[p] == 1].tolist()
    for p in pathways_of_interest
}

# --- Recall computation helper ---
def compute_recall_counts(mask, cls, probs, y_true_df):
    y_pred = probs.idxmax(axis=1)
    n_total = mask.sum()
    if n_total == 0:
        return 0.0, 0, 0
    n_correct = ((y_pred[mask] == cls) & (y_true_df[cls] == 1)).sum()
    return n_correct / n_total, n_correct, n_total

# --- Main loop ---
all_pathway_results, all_gene_results = [], []

for dataset_name, loader in datasets.items():
    for bootstrap, lc in settings:
        print(f"Processing {dataset_name} — {bootstrap} — lc{lc}")

        adata = loader(bootstrap, lc)
        gene_hom = adata.uns['gene_hom']
        y_true_df = adata.uns['pathway_hom']

        # Always SNMF variant for this bootstrap + lc
        predictions = {"SNMF": adata.uns['Yhat']}

        # LR_cosmic: add for both bootstraps but skip lc variation
        if lc == 0:  # only run once per bootstrap
            predictions[f"LR_cosmic"] = adata.uns['Yhat_LRcosmic']

        # --- Per pathway ---
        for model_name, probs in predictions.items():
            for p in pathways_of_interest:
                mask = (gene_hom[pathway_to_genes[p]].sum(axis=1) > 0) & (y_true_df[p] == 1)
                recall_val, n_correct, n_total = compute_recall_counts(mask, p, probs, y_true_df)
                all_pathway_results.append({
                    "dataset": dataset_name,
                    "bootstrap": bootstrap,
                    "lc": lc if model_name == "SNMF" else None,
                    "model": model_name,
                    "pathway": p,
                    "recall": recall_val,
                    "count": f"{n_correct}/{n_total}"
                })

        # --- Per gene ---
        for model_name, probs in predictions.items():
            for p in pathways_of_interest:
                for g in pathway_to_genes[p]:
                    mask = (gene_hom[g] > 0) & (y_true_df[p] == 1)
                    recall_val, n_correct, n_total = compute_recall_counts(mask, p, probs, y_true_df)
                    all_gene_results.append({
                        "dataset": dataset_name,
                        "bootstrap": bootstrap,
                        "lc": lc if model_name == "SNMF" else None,
                        "model": model_name,
                        "pathway": p,
                        "gene": g,
                        "recall": recall_val,
                        "count": f"{n_correct}/{n_total}"
                    })

# --- Save results ---
df_pathway = pd.DataFrame(all_pathway_results)
df_gene = pd.DataFrame(all_gene_results)
out_dir = os.path.join(project_root, "results", "reproduce", "TCGA")
os.makedirs(out_dir, exist_ok=True)
df_pathway.to_csv(os.path.join(out_dir, "recall_per_pathway.csv"), index=False)
df_gene.to_csv(os.path.join(out_dir, "recall_per_gene.csv"), index=False)


In [ ]:
# --- Generate count tables per dataset ---
df_pathway["model_variant"] = df_pathway.apply(
    lambda r: f"{r['model']}_{r['bootstrap']}_Lc{int(r['lc'])}"
              if r["model"] == "SNMF" else f"{r['model']}_{r['bootstrap']}",
    axis=1
)

for dataset in df_pathway["dataset"].unique():
    print(f"\n=== Count per Pathway per Model Variant — {dataset} ===")
    table = df_pathway[df_pathway["dataset"] == dataset].pivot_table(
        index="pathway",
        columns="model_variant",
        values="count",
        aggfunc="first"
    ).T
    print(table.to_string())

# Supplementary Analysis

In [ ]:
adata

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

adata = adata_hrd  # or adata_mmrd
# # Always reconstruct the dataframe properly
# gene_het_matrix = adata.uns['gene_het']
# # gene_het_cols = adata.uns['gene_het_cols']

# if gene_het_matrix.shape[1] != len(gene_het_cols):
#     print(f"⚠️ Mismatch! Matrix has {gene_het_matrix.shape[1]} columns, but column list has {len(gene_het_cols)} names.")

#     # OPTIONAL DEBUG
#     print("First 5 column names:", gene_het_cols[:5])
#     print("Shape of matrix:", gene_het_matrix.shape)

#     # Fix (if safe to truncate)
#     gene_het = pd.DataFrame(gene_het_matrix, index=adata.obs_names, columns=gene_het_cols[:gene_het_matrix.shape[1]])
# else:
#     gene_het = pd.DataFrame(gene_het_matrix, index=adata.obs_names, columns=gene_het_cols)




# --- Sum mutation counts across samples ---
het_counts = (adata.uns['gene_het'] > 0).sum(axis=0)
hom_counts = (adata.uns['gene_hom'] > 0).sum(axis=0)

# --- Combine into one DataFrame ---
mutation_counts = pd.DataFrame({
    "HET_count": het_counts,
    "HOM_count": hom_counts
})
mutation_counts["total"] = mutation_counts["HET_count"] + mutation_counts["HOM_count"]

# --- Filter or sort (optional) ---
# For example, keep genes with any mutations
mutation_counts = mutation_counts[mutation_counts["total"] > 0]
mutation_counts = mutation_counts.sort_values(by="total", ascending=True)

# --- Plot ---
sns.set(rc={'figure.figsize':(5,12)})
# sns.set(rc={'figure.figsize': (6, max(6, 0.35 * len(mutation_counts)))})
sns.set_style('whitegrid')

ax = mutation_counts[["HOM_count", "HET_count"]].plot(
    kind='barh',
    stacked=True,
    color=['darkred', 'darkorange']
)

# Fix bar height
for container in ax.containers:
    plt.setp(container, height=1)

# Legend and labels
plt.legend(
    frameon=True,
    title='Mutation status',
    labels=['Homozygous', 'Heterozygous'],
    loc='lower right'
)
plt.xlabel('# of tumors', fontsize=14)
plt.ylabel('Genes', fontsize=14)
plt.title('Mutation frequencies across tumors', fontsize=16)
plt.grid(axis='y')

# Optional: Save
# plt.savefig(f'/path/to/output/HR_gene_mutation_counts.svg', bbox_inches='tight')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

def plot_exposure_vs_prediction(adata, pathway_class, project_root):
    """
    Plots Yhat vs Exposure colored by mutation status for a given pathway class (e.g., 'HR', 'MMR', 'BER')
    and saves figures to {project_root}/results/figures/sup/.

    Args:
        adata (AnnData): Annotated data with pathway and exposure info
        pathway_class (str): One of ['HR', 'MMR', 'BER']
        project_root (str): Root directory of the project
    """
    sns.set(rc={'figure.figsize': (6, 4)})
    sns.set_style('whitegrid')
    col_dict = {'WT': 'silver', 'HET': 'darkorange', 'HOM': 'darkred'}

    # Output directory
    save_dir = os.path.join(project_root, "results", "figures", "sup")
    os.makedirs(save_dir, exist_ok=True)

    # Match exposure signatures that include the pathway name
    sig_cols = [col for col in adata.uns['exposures_test'].columns if pathway_class in col]
    if not sig_cols:
        print(f"⚠️ No signature exposures found for pathway: {pathway_class}")
        return

    # Compute mutation status
    status_code = (adata.uns['pathway'] + adata.uns['pathway_hom']).astype(int)
    if pathway_class not in status_code.columns:
        print(f"⚠️ Pathway '{pathway_class}' not found in adata.uns['pathway'] columns.")
        return

    status_label = np.array(['WT', 'HET', 'HOM'])[status_code[pathway_class]]

    # Check Yhat availability
    if pathway_class not in adata.uns['Yhat'].columns:
        print(f"⚠️ {pathway_class} not found in adata.uns['Yhat'].columns.")
        return

    # Plot for each relevant signature
    for sig in sig_cols:
        df = pd.DataFrame({
            'Exposure': adata.uns['exposures_test'][sig],
            'Yhat': adata.uns['Yhat'][pathway_class],
            'Status': pd.Categorical(status_label, categories=['WT', 'HET', 'HOM'])
        })

        plt.figure()
        sns.scatterplot(
            data=df, x='Exposure', y='Yhat', hue='Status',
            palette=col_dict, s=30, alpha=0.8
        )
        plt.xlabel(f"{sig} (exposure)", fontsize=14)
        plt.ylabel(f"$Y_{{{pathway_class}}}$", fontsize=14)
        plt.legend(frameon=True, title=f"{pathway_class} status", loc='lower right')
        plt.tight_layout()

        save_path = os.path.join(save_dir, f"decision_Y_E_{pathway_class}_{sig}.svg")
        plt.savefig(save_path, bbox_inches='tight')
        plt.show()

        print(f"✅ Saved plot: {save_path}")


# Example usage
plot_exposure_vs_prediction(adata_hrd, 'HR', project_root)
plot_exposure_vs_prediction(adata_mmrd, 'MMR', project_root)


## Multi-Deficiency Analysis

In [ ]:
# print(adata_hrd.uns['pathway_cols'])
print(adata_hrd.uns['pathway'].sum(axis=0))
print(adata_hrd.uns['pathway_hom'].sum(axis=0))


In [ ]:
import numpy as np
import pandas as pd

def check_multi_deficiency(adata, ddr_labels=('MMR','HR','BER'), label_type='pathway_hom'):
    """
    Check if any samples have multiple deficiencies among selected DDR classes.

    Works with either:
      - adata.obsm[label_type] = ndarray, and adata.uns[f'{label_type}_cols'] = list of column names
      - adata.uns[label_type]  = DataFrame with columns named as classes

    Parameters
    ----------
    adata : AnnData
    ddr_labels : iterable of str
        Subset of class names to check (e.g., ('MMR','HR','BER'))
    label_type : {'pathway','pathway_hom'}
        Which label matrix to use.
    """

    # ---- Resolve matrix (M) and its column names (cols) ----
    M, cols = None, None

    # Case A: matrix in obsm, names in uns['..._cols']
    if label_type in adata.obsm:
        M = adata.obsm[label_type]
        cols = adata.uns.get(f"{label_type}_cols") \
               or adata.uns.get("pathway_cols") \
               or adata.uns.get("pathway_hom_cols")

    # Case B: full DataFrame in uns[label_type]
    if M is None and label_type in adata.uns:
        obj = adata.uns[label_type]
        if isinstance(obj, pd.DataFrame):
            # align rows to adata.obs order if possible
            df = obj.reindex(index=adata.obs_names, fill_value=0)
            cols = list(df.columns)
            M = df.values
        elif isinstance(obj, np.ndarray):
            M = obj
            cols = adata.uns.get(f"{label_type}_cols")

    if M is None or cols is None:
        raise ValueError(
            f"Could not find '{label_type}' labels. "
            f"Expected either adata.obsm['{label_type}'] with adata.uns['{label_type}_cols'], "
            f"or adata.uns['{label_type}'] as a DataFrame with columns."
        )

    # ---- Index for selected DDR classes (ignore labels not present) ----
    cols = list(cols)
    missing = [c for c in ddr_labels if c not in cols]
    if missing:
        print(f"[warn] Missing labels (ignored): {missing}")
    keep = [c for c in ddr_labels if c in cols]
    if not keep:
        raise ValueError("None of the requested ddr_labels are present in the label matrix.")

    idx = [cols.index(c) for c in keep]
    sub = M[:, idx]

    # ---- Count multi-deficient samples ----
    row_sums = np.asarray(sub).sum(axis=1)
    multi_def_mask = row_sums > 1

    n_multi = int(multi_def_mask.sum())
    print(f"[{label_type}] samples with multiple deficiencies in {keep}: {n_multi} / {adata.n_obs}")
    if n_multi > 0:
        print("Sample IDs:", adata.obs_names[multi_def_mask].tolist())

# Any mutation (het + hom)
check_multi_deficiency(adata_hrd, label_type='pathway')
# Homozygous only
check_multi_deficiency(adata_hrd, label_type='pathway_hom')

check_multi_deficiency(adata_mmrd, label_type='pathway')
check_multi_deficiency(adata_mmrd, label_type='pathway_hom')



In [ ]:
def plot_hr_vs_mmr_exposure(
    adata,
    hr_sigs=('SBS3','SBS8'),
    mmr_sigs=("SBS6","SBS14","SBS15","SBS20","SBS21","SBS26","SBS44"),
    sig_type='cosmic',          # 'exposure' or 'cosmic'
    color_by='multi_deficiency',# 'pathway','pathway_hom','multi_deficiency', or None
    color_by_gene=None,         # e.g. 'MLH1' (overrides color_by)
    ddr_classes=('MMR','HR','BER'),
    alpha=0.7,
    figsize=(7,6),
    verbose=False
):
    import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

    # ---- Resolve exposures (matrix + colnames) ----
    def _resolve_matrix(name):
        # Preferred: obsm[name] + uns[f"{name}_cols"]
        if name in adata.obsm:
            mat = adata.obsm[name]
            cols = adata.uns.get(f"{name}_cols")
            if cols is None:
                raise ValueError(f"Missing adata.uns['{name}_cols'] for obsm['{name}'].")
            return mat, list(cols)
        # Fallback: uns[name] as a DataFrame with columns
        if name in adata.uns:
            obj = adata.uns[name]
            if isinstance(obj, pd.DataFrame):
                df = obj.reindex(index=adata.obs_names, fill_value=0)
                return df.values, list(df.columns)
        raise ValueError(f"Could not find exposures for '{name}'. "
                         f"Expected adata.obsm['{name}'] with adata.uns['{name}_cols'] "
                         f"or adata.uns['{name}'] as a DataFrame.")

    X, cols = _resolve_matrix('exposures_test' if sig_type=='exposure' else 'cosmic_exposures_norm')

    # ---- Sum selected signatures ----
    def _sum_signatures(sig_list):
        try:
            idx = [cols.index(s) for s in sig_list]
        except ValueError as e:
            raise ValueError(f"Signature not found: {e}")
        return np.asarray(X)[:, idx].sum(axis=1)

    mmr_sum = _sum_signatures(mmr_sigs)
    hr_sum  = _sum_signatures(hr_sigs)

    df = pd.DataFrame({"MMR": mmr_sum, "HR": hr_sum}, index=adata.obs_names)

    # ---- Coloring rules ----
    if color_by_gene:  # overrides color_by
        gene = color_by_gene
        # Resolve gene matrices (het/hom)
        if 'gene_het' not in adata.obsm or 'gene_hom' not in adata.obsm:
            raise ValueError("Need adata.obsm['gene_het'] and adata.obsm['gene_hom'].")
        het_cols = adata.uns.get('gene_het_cols'); hom_cols = adata.uns.get('gene_hom_cols')
        if het_cols is None or hom_cols is None:
            raise ValueError("Need adata.uns['gene_het_cols'] and adata.uns['gene_hom_cols'].")
        if gene not in het_cols or gene not in hom_cols:
            raise ValueError(f"Gene '{gene}' not found in gene_het_cols/gene_hom_cols.")

        hi, gi = het_cols.index(gene), hom_cols.index(gene)
        het = adata.obsm['gene_het'][:, hi]; hom = adata.obsm['gene_hom'][:, gi]

        def _class(het_i, hom_i):
            return 'HOM' if hom_i==1 else ('HET' if het_i==1 else 'WT')

        df['color_label'] = [ _class(h,g) for h,g in zip(het, hom) ]
        palette = {'WT':'#9aa0a6','HET':'#fb8c00','HOM':'#d32f2f'}
        legend_title = gene

    elif color_by in ('pathway','pathway_hom'):
        # binary label for a single class in ddr_classes
        if len(ddr_classes) != 1:
            raise ValueError("For color_by 'pathway' or 'pathway_hom', provide a single class in ddr_classes.")
        cls = ddr_classes[0]
        # Resolve the matrix + cols just like exposures
        Y, ycols = _resolve_matrix(color_by)
        if cls not in ycols:
            raise ValueError(f"'{cls}' not in {color_by} columns.")
        df['color_label'] = Y[:, ycols.index(cls)]
        palette = {0:'#9aa0a6', 1:'#d32f2f'}
        legend_title = cls

    elif color_by == 'multi_deficiency':
        # Sum selected DDR columns from adata.obsm['pathway']
        Y, ycols = _resolve_matrix('pathway')
        missing = [c for c in ddr_classes if c not in ycols]
        if missing:
            raise ValueError(f"Missing DDR classes in 'pathway': {missing}")
        idx = [ycols.index(c) for c in ddr_classes]
        counts = np.asarray(Y)[:, idx].sum(axis=1)

        def _lab(c):
            return 'WT' if c==0 else ('Single' if c==1 else 'Multi')

        df['color_label'] = [ _lab(int(c)) for c in counts ]
        palette = {'WT':'#9aa0a6','Single':'#fb8c00','Multi':'#d32f2f'}
        legend_title = "Deficiency"

    else:
        df['color_label'] = 'All'
        palette = {'All':'#9aa0a6'}
        legend_title = None

    # ---- Verbose: highlight high-exposure samples (both >20) ----
    if verbose:
        mask = (df['MMR']>20) & (df['HR']>20)
        if mask.any():
            print("Samples with exposure > 20 in both MMR and HR:")
            for s in df.index[mask]:
                print("  -", s)

    # ---- Plot ----
    plt.figure(figsize=figsize)
    ax = sns.scatterplot(data=df, x="MMR", y="HR", hue="color_label", palette=palette, alpha=alpha, edgecolor=None)
    ax.set_xlabel(f"Sum of MMR signatures ({', '.join(mmr_sigs)})")
    ax.set_ylabel(f"Sum of HR signatures ({', '.join(hr_sigs)})")
    ax.set_title("HR vs MMR Signature Exposure")
    if legend_title:
        ax.legend(title=legend_title)
    else:
        ax.get_legend().remove()
    plt.tight_layout(); plt.show()

    return df.index[(df['MMR']>20) & (df['HR']>20)].tolist()

def get_sample_details(adata, sample_names, verbose=True):
    """
    For given sample names, extract exposures and mutated genes using DataFrames
    stored in `adata.uns`. Falls back to `obsm` + *_cols if needed.

    Expected (preferred) keys in `adata.uns`:
      - 'exposures_test' : pd.DataFrame (index = samples, columns = signatures)
      - 'gene_het'       : pd.DataFrame (index = samples, columns = genes)
      - 'gene_hom'       : pd.DataFrame (index = samples, columns = genes)
    """

    import pandas as pd

    # --- validate samples ---
    missing = [s for s in sample_names if s not in adata.obs_names]
    if missing:
        raise ValueError(f"Samples not in adata.obs_names: {missing}")

    # --- helpers: pull DF from .uns, else rebuild from .obsm + cols ---
    def _get_df(key_df, key_mat, key_cols):
        if key_df in adata.uns and isinstance(adata.uns[key_df], pd.DataFrame):
            df = adata.uns[key_df]
            # ensure all requested samples exist as rows
            return df.reindex(index=adata.obs_names).loc[sample_names]
        # fallback
        if key_mat in adata.obsm and key_cols in adata.uns:
            mat = adata.obsm[key_mat]
            cols = adata.uns[key_cols]
            df = pd.DataFrame(mat, index=adata.obs_names, columns=cols)
            return df.loc[sample_names]
        raise ValueError(
            f"Need either adata.uns['{key_df}'] as DataFrame, "
            f"or adata.obsm['{key_mat}'] + adata.uns['{key_cols}']."
        )

    # exposures
    exposure_df = _get_df("exposures_test", "exposures_test", "exposures_test_cols")

    # gene het / hom
    het_df = _get_df("gene_het", "gene_het", "gene_het_cols")
    hom_df = _get_df("gene_hom", "gene_hom", "gene_hom_cols")

    # --- extract mutated gene lists ---
    def _genes_from_row(row):  # row is 0/1 (or bool)
        return row.index[row.astype(bool)].tolist()

    het_genes = het_df.apply(_genes_from_row, axis=1)
    hom_genes = hom_df.apply(_genes_from_row, axis=1)

    # --- assemble result ---
    # store exposures as dicts for readability
    exposures_dicts = [exposure_df.loc[s].to_dict() for s in sample_names]

    results = pd.DataFrame(
        {
            "exposures": exposures_dicts,
            "het_genes": het_genes.values,
            "hom_genes": hom_genes.values,
        },
        index=sample_names,
    )

    if verbose:
        for s in sample_names:
            print(f"\nSample: {s}")
            print("  Exposures:")
            for sig, val in results.loc[s, "exposures"].items():
                print(f"    {sig}: {val:.2f}")
            print(f"  Het Mutations ({len(results.loc[s, 'het_genes'])}): {results.loc[s, 'het_genes']}")
            print(f"  Hom Mutations ({len(results.loc[s, 'hom_genes'])}): {results.loc[s, 'hom_genes']}")

    return results


In [ ]:
samples = plot_hr_vs_mmr_exposure(
    adata_mmrd,
    hr_sigs=['SBS3', 'SBS8'],
    mmr_sigs=["SBS6", "SBS14", "SBS15", "SBS20", "SBS21", "SBS26", "SBS44"],
    sig_type='cosmic',
    color_by='multi_deficiency',
    ddr_classes=['MMR', 'HR', 'BER'],
    verbose=True
)

details_df = get_sample_details(adata_mmrd, samples, verbose=True)


In [ ]:
samples = plot_hr_vs_mmr_exposure(
    adata_mmrd,
    hr_sigs=['SBS6'],
    mmr_sigs=['SBS15'],
    sig_type='cosmic',
    color_by='multi_deficiency',  # fallback
    # color_by_gene='MLH1',         # takes priority
    ddr_classes=['MMR'],
    verbose=True
)


details_df = get_sample_details(adata_mmrd, samples, verbose=True)


## SNMF BER-UNG exposure vs COSMIC SBS30 (UNGd related)

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import linregress

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)

# Choose which exposure matrix to use
exposure_key = "exposures_test"

# ------------------------------------------------------------------
# Combine adata_hrd and adata_mmrd
# ------------------------------------------------------------------
def extract_df(adata):
    X_dn = pd.DataFrame(adata.uns[exposure_key], index=adata.obs_names)
    X_cos = pd.DataFrame(adata.uns["cosmic_exposures_norm"], index=adata.obs_names)
    PW = pd.DataFrame(adata.uns["pathway"], index=adata.obs_names)
    PW_hom = pd.DataFrame(adata.uns["pathway_hom"], index=adata.obs_names)

    # Find BER/UNG signature
    ung_cols = [c for c in X_dn.columns if "_BER_UNG" in c or c.endswith("BER_UNG")]
    if not ung_cols:
        raise ValueError(f"No BER/UNG-like signature found in {exposure_key}")
    ung_col = ung_cols[0]

    df = pd.DataFrame({
        "x": X_dn[ung_col],
        "y": X_cos["SBS30"],
        "ber": PW["BER"],
        "ber_hom": PW_hom["BER"]
    }, index=adata.obs_names)
    df["source"] = adata.uns.get("adata_type", "adata")
    return df, ung_col

df_hrd, ung_col = extract_df(adata_hrd)
df_mmrd, _ = extract_df(adata_mmrd)

df = pd.concat([df_hrd, df_mmrd], axis=0)

# classify mutation status
def classify(row):
    if row["ber_hom"] == 1:
        return "Hom"
    elif row["ber"] == 1:
        return "Het"
    else:
        return "WT"

df["status"] = df.apply(classify, axis=1)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["x", "y"])

# ------------------------------------------------------------------
# Plot: scatter + marginals
# ------------------------------------------------------------------
palette = {"WT": "grey", "Het": "orange", "Hom": "red"}
plot_order = ["WT", "Het", "Hom"]

pearson_r = df["x"].corr(df["y"], method="pearson")
slope, intercept, _, _, _ = linregress(df["x"], df["y"])

fig = plt.figure(figsize=(7, 7))
gs = gridspec.GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                       wspace=0.05, hspace=0.05)

ax_scatter = plt.subplot(gs[1, 0])
ax_xdist   = plt.subplot(gs[0, 0], sharex=ax_scatter)
ax_ydist   = plt.subplot(gs[1, 1], sharey=ax_scatter)

# scatter (WT→Het→Hom so red is on top)
for status in plot_order:
    subset = df[df["status"] == status]
    sns.scatterplot(
        data=subset, x="x", y="y", label=status,
        color=palette[status], edgecolor="none",
        s=50, alpha=0.5, ax=ax_scatter
    )

# regression line
x_vals = np.linspace(df["x"].min(), df["x"].max(), 100)
ax_scatter.plot(x_vals, slope * x_vals + intercept, color="black", lw=2)

ax_scatter.set_xlabel(f"Exposure: {ung_col}")
ax_scatter.set_ylabel("COSMIC exposure: SBS30")
# ax_scatter.set_ylim(0, 1)

# legend and correlation text
handles, labels = ax_scatter.get_legend_handles_labels()
ax_scatter.legend(handles[::-1], labels[::-1], title="BER status", frameon=False)
ax_scatter.text(0.02, 0.98, f"Pearson r = {pearson_r:.2f}",
                transform=ax_scatter.transAxes, va="top", ha="left")

# marginal boxplots
sns.boxplot(data=df, x="x", hue="status", palette=palette, ax=ax_xdist, linewidth=1, fliersize=2)
ax_xdist.axis("off")
if ax_xdist.get_legend() is not None:
    ax_xdist.get_legend().remove()

sns.boxplot(data=df, y="y", hue="status", palette=palette, ax=ax_ydist, linewidth=1, fliersize=2)
ax_ydist.axis("off")
if ax_ydist.get_legend() is not None:
    ax_ydist.get_legend().remove()

# style and borders
for ax in [ax_scatter, ax_xdist, ax_ydist]:
    ax.set_facecolor("white")
    ax.grid(True, color="lightgrey", linestyle="--", linewidth=0.5, alpha=0.5)
for spine in ax_scatter.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.2)

plt.tight_layout()
save_path = os.path.join(fig_dir, f"UNG_SBS30_scatter_marginals_combined_{exposure_key}.pdf")
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"✅ Saved: {save_path}")
plt.show()

# ------------------------------------------------------------------
# Boxplot of BER/UNG exposure per status (y 0–1)
# ------------------------------------------------------------------
plt.figure(figsize=(5,4))
sns.boxplot(data=df, x="status", y="x", palette=palette, order=plot_order, linewidth=1.2, fliersize=2)
sns.stripplot(data=df, x="status", y="x", color="black", alpha=0.3, size=3, order=plot_order)
plt.ylim(0, 1)
plt.xlabel("BER status")
plt.ylabel(f"Exposure: {ung_col}")
plt.title("BER-UNG exposure per mutation status (combined HRD + MMRD)")
plt.tight_layout()

box_path = os.path.join(fig_dir, f"UNG_boxplot_combined_{exposure_key}.pdf")
plt.savefig(box_path, bbox_inches="tight", dpi=300)
print(f"✅ Saved: {box_path}")
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import linregress

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)

adata = adata_mmrd  # or adata_hrd
adata_label = adata.uns.get("adata_type", "adata")

# ------------------------------------------------------------------
# Build dataframe
# ------------------------------------------------------------------
X_dn   = pd.DataFrame(adata.uns["exposures_test"], index=adata.obs_names)
X_cos  = pd.DataFrame(adata.uns["cosmic_exposures_norm"], index=adata.obs_names)
gene_het = pd.DataFrame(adata.uns["gene_het"], index=adata.obs_names)
gene_hom = pd.DataFrame(adata.uns["gene_hom"], index=adata.obs_names)

df = pd.DataFrame({
    "x": X_dn["SBS96D_BER_UNG"],
    "y": X_cos["SBS30"],
    "UNG_het": gene_het["UNG"],
    "UNG_hom": gene_hom["UNG"]
}, index=adata.obs_names)

# Classify mutation status
def classify(row):
    if row["UNG_hom"] == 1:
        return "Hom"
    elif row["UNG_het"] == 1:
        return "Het"
    else:
        return "WT"

df["status"] = df.apply(classify, axis=1)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["x", "y"])

palette = {"WT": "grey", "Het": "orange", "Hom": "red"}
plot_order = [s for s in ["WT", "Het", "Hom"] if s in df["status"].unique()]

# ------------------------------------------------------------------
# Scatter with marginals
# ------------------------------------------------------------------
pearson_r = df["x"].corr(df["y"], method="pearson")
slope, intercept, _, _, _ = linregress(df["x"], df["y"])

fig = plt.figure(figsize=(7, 7))
gs = gridspec.GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                       wspace=0.05, hspace=0.05)
ax_scatter = plt.subplot(gs[1, 0])
ax_xdist   = plt.subplot(gs[0, 0], sharex=ax_scatter)
ax_ydist   = plt.subplot(gs[1, 1], sharey=ax_scatter)

# Scatter plot
for status in plot_order:
    subset = df[df["status"] == status]
    sns.scatterplot(
        data=subset, x="x", y="y", label=status,
        color=palette[status], edgecolor="none",
        s=50, alpha=0.6, ax=ax_scatter
    )

# Regression line
x_vals = np.linspace(df["x"].min(), df["x"].max(), 100)
ax_scatter.plot(x_vals, slope * x_vals + intercept, color="black", lw=2)

# Axis labels
ax_scatter.set_xlabel("De-novo exposure: SBS96D_BER_UNG")
ax_scatter.set_ylabel("COSMIC exposure: SBS30")
# ax_scatter.set_ylim(0, 1)

# Legend and r text
ax_scatter.legend(title="UNG status", frameon=False)
ax_scatter.text(0.02, 0.98, f"Pearson r = {pearson_r:.2f}",
                transform=ax_scatter.transAxes, va="top", ha="left")

# --- Marginals (robust: Matplotlib boxplots, no seaborn hue bug) ---
# Top: horizontal boxes of x by status
groups_x = [df.loc[df["status"] == s, "x"].values for s in plot_order]
bp_x = ax_xdist.boxplot(groups_x, vert=False, positions=np.arange(len(plot_order)),
                        patch_artist=True, widths=0.6, whis=1.5)
for i, patch in enumerate(bp_x["boxes"]):
    patch.set_facecolor(palette[plot_order[i]])
    patch.set_alpha(0.7)
    patch.set_edgecolor("black")
for k in ["caps", "whiskers", "medians"]:
    for line in bp_x[k]:
        line.set_color("black")
        line.set_linewidth(1)

ax_xdist.set_yticks([])         # hide y ticks
ax_xdist.set_xticks([])         # hide x ticks
ax_xdist.axis("off")

# Right: vertical boxes of y by status
groups_y = [df.loc[df["status"] == s, "y"].values for s in plot_order]
bp_y = ax_ydist.boxplot(groups_y, vert=True, positions=np.arange(len(plot_order)),
                        patch_artist=True, widths=0.6, whis=1.5)
for i, patch in enumerate(bp_y["boxes"]):
    patch.set_facecolor(palette[plot_order[i]])
    patch.set_alpha(0.7)
    patch.set_edgecolor("black")
for k in ["caps", "whiskers", "medians"]:
    for line in bp_y[k]:
        line.set_color("black")
        line.set_linewidth(1)

ax_ydist.set_xticks([])         # hide x ticks
ax_ydist.set_yticks([])         # hide y ticks
ax_ydist.axis("off")


# Styling
for ax in [ax_scatter, ax_xdist, ax_ydist]:
    ax.set_facecolor("white")
    ax.grid(True, color="lightgrey", linestyle="--", linewidth=0.5, alpha=0.5)
for spine in ax_scatter.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.2)

plt.tight_layout()
save_path = os.path.join(fig_dir, f"UNG_SBS30_scatter_marginals_{adata_label}.pdf")
plt.savefig(save_path, bbox_inches="tight", dpi=300)
print(f"✅ Saved scatter + marginals: {save_path}")
plt.show()

# ------------------------------------------------------------------
# Boxplot of UNG exposure per mutation status
# ------------------------------------------------------------------
plt.figure(figsize=(5, 4))
sns.boxplot(data=df, x="status", y="x", palette=palette,
            order=plot_order, linewidth=1.2, fliersize=2)
sns.stripplot(data=df, x="status", y="x", color="black",
              alpha=0.3, size=3, order=plot_order)
plt.ylim(0, 1)
plt.xlabel("UNG status")
plt.ylabel("Exposure: SBS96D_BER_UNG")
plt.title(f"UNG exposure per mutation status ({adata_label})")
plt.tight_layout()

box_path = os.path.join(fig_dir, f"UNG_boxplot_{adata_label}.pdf")
plt.savefig(box_path, bbox_inches="tight", dpi=300)
print(f"✅ Saved boxplot: {box_path}")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

def _get_df(adata, key_df, key_mat=None, key_cols=None):
    """Prefer a named DataFrame in .uns; fall back to .obsm + cols if needed."""
    if key_df in adata.uns and isinstance(adata.uns[key_df], pd.DataFrame):
        return adata.uns[key_df].reindex(index=adata.obs_names)
    if key_mat and key_cols and key_mat in adata.obsm and key_cols in adata.uns:
        mat = adata.obsm[key_mat]
        cols = adata.uns[key_cols]
        return pd.DataFrame(mat, index=adata.obs_names, columns=cols)
    raise ValueError(f"Need adata.uns['{key_df}'] as DataFrame, "
                     f"or adata.obsm['{key_mat}'] + adata.uns['{key_cols}'].")

# adata = adata_hrd
adata = adata_mmrd

# --- saving paths ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
adata_label = adata.uns.get('adata_type', 'adata')

# --- 1) Load Yhat as DataFrame (samples x pathways) ---
Yhat = _get_df(adata, "Yhat", key_mat="Yhat", key_cols="Yhat_cols")  # flexible

# --- 2) Distribution of all Yhat values ---
plt.figure(figsize=(5,3))
plt.hist(Yhat.values.ravel(), bins=40, edgecolor='none', alpha=0.9)
plt.xlabel("Yhat value"); plt.ylabel("Count"); plt.title("Distribution of Yhat (all entries)")
plt.tight_layout()

save_path = os.path.join(fig_dir, f"yhat_distribution_{adata_label}.pdf")
plt.savefig(save_path, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

# --- 3) For each sample: max and second highest Yhat ---
sorted_idx = np.argsort(-Yhat.values, axis=1)
top1_idx = sorted_idx[:, 0]
top2_idx = sorted_idx[:, 1]
top1_vals = Yhat.values[np.arange(Yhat.shape[0]), top1_idx]
top2_vals = Yhat.values[np.arange(Yhat.shape[0]), top2_idx]
top1_lab = Yhat.columns.values[top1_idx]
top2_lab = Yhat.columns.values[top2_idx]

rank2_df = pd.DataFrame({
    "top1": top1_lab, "top1_val": top1_vals,
    "top2": top2_lab, "top2_val": top2_vals,
    "mutation_count": adata.uns['mutation_count']
}, index=Yhat.index)

# --- 4) Heterozygous pathway calls (samples x pathways) ---
if "pathway_het" in adata.uns and isinstance(adata.uns["pathway_het"], pd.DataFrame):
    P_het = adata.uns["pathway_het"].reindex(index=adata.obs_names)
else:
    P_any = _get_df(adata, "pathway",      key_mat="pathway",      key_cols="pathway_cols")
    if "pathway_hom" in adata.uns or ("pathway_hom" in adata.obsm and "pathway_hom_cols" in adata.uns):
        P_hom = _get_df(adata, "pathway_hom", key_mat="pathway_hom", key_cols="pathway_hom_cols")
        P_het = (P_any - P_hom).clip(lower=0).astype(int)
    else:
        P_het = (P_any > 0).astype(int)

P_het = P_het.reindex(columns=Yhat.columns).fillna(0).astype(int)

both_present = []
for s, r in rank2_df.iterrows():
    a, b = r["top1"], r["top2"]
    pa = P_het.at[s, a] if a in P_het.columns else 0
    pb = P_het.at[s, b] if b in P_het.columns else 0
    both_present.append(int((pa == 1) and (pb == 1)))
rank2_df["both_top2_present_het"] = both_present

# --- Scatter: max vs. second max; color by both_top2_present_het ---
plt.figure(figsize=(7,4))
mask = rank2_df["both_top2_present_het"].astype(bool)
scale = 500 / rank2_df["mutation_count"].max()
sizes = rank2_df["mutation_count"] * scale

plt.scatter(rank2_df.loc[~mask, "top1_val"], rank2_df.loc[~mask, "top2_val"],
            s=sizes[~mask], alpha=0.7, label="Not both present (het)")
plt.scatter(rank2_df.loc[mask, "top1_val"], rank2_df.loc[mask, "top2_val"],
            s=sizes[mask], alpha=0.9, label="Both present (het)")

plt.plot([0,1],[0,1], ls="--", lw=1, color="gray")
plt.xlabel("Max Yhat"); plt.ylabel("2nd-highest Yhat")
plt.title("Max vs. 2nd-highest Yhat per sample")
leg1 = plt.legend(frameon=False, loc="upper left")

sizes_for_legend = [250, 2500, 25000]
handles = [
    Line2D([], [], marker='o', linestyle='None',
           markersize=np.sqrt(v * scale),
           color='gray', alpha=0.5, label=f"{v}")
    for v in sizes_for_legend
]
plt.legend(handles=handles, title="Mutation count",
           bbox_to_anchor=(1.05, 0.5), loc="center left",
           frameon=False, labelspacing=1.2)
plt.gca().add_artist(leg1)

plt.tight_layout()

save_path = os.path.join(fig_dir, f"yhat_top1_vs_top2_{adata_label}.pdf")
plt.savefig(save_path, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

# --- Ambiguous list ---
amb_thresh = 0.25
ambiguous = rank2_df[(rank2_df["top2_val"] >= amb_thresh) & (rank2_df["both_top2_present_het"] == 1)]
print(f"\nAmbiguous samples (2nd-highest ≥ {amb_thresh} and both top-2 present as het): {len(ambiguous)}")
display(ambiguous.sort_values("top2_val", ascending=False).head(20))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches


# --- Helpers ---
def _get_df(adata, key_df, key_mat=None, key_cols=None):
    """Prefer a DataFrame in .uns; fallback to .obsm + names in .uns."""
    if key_df in adata.uns and isinstance(adata.uns[key_df], pd.DataFrame):
        return adata.uns[key_df].reindex(index=adata.obs_names)
    if key_mat and key_cols and key_mat in adata.obsm and key_cols in adata.uns:
        mat = adata.obsm[key_mat]
        cols = adata.uns[key_cols]
        return pd.DataFrame(mat, index=adata.obs_names, columns=cols)
    raise ValueError(f"Need adata.uns['{key_df}'] as DataFrame, "
                     f"or adata.obsm['{key_mat}'] + adata.uns['{key_cols}'].")

def _parse_pathway_from_sig(colname: str):
    # e.g., SBS96A_HR -> 'HR'; SBS96D_BER_UNG -> 'BER'; SBS96C_Control -> 'Control'
    toks = str(colname).split("_")
    return toks[1] if len(toks) >= 2 else "Unknown"



adata = adata_hrd  
# adata = adata_mmrd

# --- 1) Load exposures (samples x signatures) ---
EX = _get_df(adata, "exposures_test", key_mat="exposures_test", key_cols="exposures_test_cols")

# --- 2) Map each signature to a pathway and aggregate per pathway ---
sig_to_pw = {c: _parse_pathway_from_sig(c) for c in EX.columns}
desired = ["HR", "MMR", "BER", "Control"]  # Control optional; kept if present
present = [p for p in desired if p in set(sig_to_pw.values())]

# Build per-pathway totals
pw_totals = {
    pw: EX.loc[:, [c for c in EX.columns if sig_to_pw[c] == pw]].sum(axis=1)
    for pw in present
}
PW = pd.DataFrame(pw_totals, index=EX.index)  # samples x pathways

# --- 3) Plot distribution of all signature exposures ---
plt.figure(figsize=(5,3))
plt.hist(EX.values.ravel(), bins=40, edgecolor='none', alpha=0.9)
plt.xlabel("Signature exposure"); plt.ylabel("Count")
plt.title("Distribution of all signature exposures")
plt.tight_layout(); plt.show()

# --- 4) For each sample: top1 vs top2 among {HR, MMR, BER[, Control]} ---
vals = PW.values
cols = PW.columns.to_numpy()
if vals.shape[1] < 2:
    raise ValueError("Need ≥2 pathway groups (e.g., HR/MMR/BER) mapped from exposures.")

sorted_idx = np.argsort(-vals, axis=1)  # descending
top1_idx = sorted_idx[:, 0]
top2_idx = sorted_idx[:, 1]
top1_vals = vals[np.arange(vals.shape[0]), top1_idx]
top2_vals = vals[np.arange(vals.shape[0]), top2_idx]
top1_lab  = cols[top1_idx]
top2_lab  = cols[top2_idx]

rank2_exp = pd.DataFrame({
    "top1": top1_lab, "top1_val": top1_vals,
    "top2": top2_lab, "top2_val": top2_vals,
    "mutation_count": adata.uns['mutation_count']
}, index=PW.index)

# --- 5) Heterozygous pathway calls (to flag multi-deficiency across top-2) ---
# Prefer a DataFrame in .uns['pathway_het']; else derive het = any - hom (clipped)
if "pathway_het" in adata.uns and isinstance(adata.uns["pathway_het"], pd.DataFrame):
    P_het = adata.uns["pathway_het"].reindex(index=adata.obs_names)
else:
    P_any = _get_df(adata, "pathway",      key_mat="pathway",      key_cols="pathway_cols")
    if ("pathway_hom" in adata.uns and isinstance(adata.uns["pathway_hom"], pd.DataFrame)) \
       or ("pathway_hom" in adata.obsm and "pathway_hom_cols" in adata.uns):
        P_hom = _get_df(adata, "pathway_hom", key_mat="pathway_hom", key_cols="pathway_hom_cols")
        P_het = (P_any - P_hom).clip(lower=0).astype(int)
    else:
        P_het = (P_any > 0).astype(int)

# Align pathway columns to PW’s pathway set (add missing as 0)
P_het = P_het.reindex(columns=list(cols)).fillna(0).astype(int)

# Flag if both top-2 pathways are heterozygous-present
both_present = []
for s, r in rank2_exp.iterrows():
    a, b = r["top1"], r["top2"]
    pa = P_het.at[s, a] if a in P_het.columns else 0
    pb = P_het.at[s, b] if b in P_het.columns else 0
    both_present.append(int((pa == 1) and (pb == 1)))
rank2_exp["both_top2_present_het"] = both_present

plt.figure(figsize=(6,4))
mask = rank2_exp["both_top2_present_het"].astype(bool)
# --- Robust size scaling (shared across both groups) ---
mc = rank2_exp["mutation_count"].fillna(0).astype(float)
mc_max = mc.max()
# Avoid division by zero; cap a minimum visual size
base_area = 200.0
scale = base_area / mc_max if mc_max > 0 else 1.0
sizes = (mc * scale).clip(lower=5)  # ensure minimum visible size

# --- Two scatters (use identical 'sizes' for both masks) ---
plt.scatter(rank2_exp.loc[~mask, "top1_val"],
            rank2_exp.loc[~mask, "top2_val"],
            s=sizes.loc[~mask].values,
            alpha=0.7, label="Not both present (het)")

plt.scatter(rank2_exp.loc[mask,  "top1_val"],
            rank2_exp.loc[mask,  "top2_val"],
            s=sizes.loc[mask].values,
            alpha=0.9, label="Both present (het)")

# Diagonal reference
max_val = float(max(rank2_exp["top1_val"].max(), rank2_exp["top2_val"].max()))
plt.plot([0, max_val], [0, max_val], ls="--", lw=1, color="gray")

plt.xlabel("Top pathway exposure")
plt.ylabel("2nd pathway exposure")
plt.title("Top vs. 2nd pathway exposure per sample")

# Left legend = category legend
leg1 = plt.legend(frameon=False, loc="upper left", title=None)

# --- Mutation-count size legend on the right (use Line2D proxies) ---
sizes_for_legend = [250, 2500, 25000]
handles = [
    Line2D([], [], marker='o', linestyle='None',
           markersize=np.sqrt(v * scale),  # sqrt because scatter 's' is area in points^2
           color='gray', alpha=0.5, label=f"{v}")
    for v in sizes_for_legend
]
plt.legend(handles=handles, title="Mutation count",
           bbox_to_anchor=(1.05, 0.5), loc="center left",
           frameon=False, labelspacing=1.2)
plt.gca().add_artist(leg1)

plt.tight_layout()
plt.show()

# --- 7) List “ambiguous” by exposure: large 2nd pathway and both present ---
amb_thresh = np.percentile(rank2_exp["top2_val"], 75)  # tweak threshold if desired
amb_exp = rank2_exp[(rank2_exp["top2_val"] >= amb_thresh) &
                    (rank2_exp["both_top2_present_het"] == 1)]
print(f"\nExposure-based ambiguous samples (2nd-highest ≥ {amb_thresh:.2f} and both top-2 present as het): {len(amb_exp)}")
display(amb_exp.sort_values("top2_val", ascending=False).head(20))


In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

adata = adata_hrd
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
adata_label = adata.uns.get('adata_type', 'adata')

snmf_exposures = pd.DataFrame(adata.uns["exposures_test"],        index=adata.obs_names)
cosmic_exposures = pd.DataFrame(adata.uns["cosmic_exposures_norm"], index=adata.obs_names)

common_idx = snmf_exposures.index.intersection(cosmic_exposures.index)
snmf_exposures = snmf_exposures.loc[common_idx]
cosmic_exposures = cosmic_exposures.loc[common_idx]

similarity = pd.DataFrame(
    cosine_similarity(snmf_exposures.T, cosmic_exposures.T),
    index=snmf_exposures.columns,
    columns=cosmic_exposures.columns
)

cosmic_sums = cosmic_exposures.sum(axis=0)
snmf_sums = snmf_exposures.sum(axis=0)

fig = plt.figure(figsize=(25, 7))
gs = gridspec.GridSpec(2, 3, width_ratios=[20, 2, 1], height_ratios=[15, 5], wspace=0.05, hspace=0.05)

ax_top = plt.subplot(gs[0, 0])
ax_top.bar(cosmic_sums.index, cosmic_sums.values, color="gray")
ax_top.set_xlim(-0.5, len(cosmic_sums) - 0.5)
ax_top.set_xticks([])
ax_top.set_ylabel("Total Exposure")
ax_top.tick_params(axis="x", bottom=False)
ax_top.set_title("SNMF vs COSMIC Exposure Similarity (Cosine)", pad=20)

ax_right = plt.subplot(gs[1, 1])
ax_right.barh(range(len(snmf_sums)), snmf_sums.values, color="gray")
ax_right.set_ylim(len(snmf_sums) - 0.5, -0.5)
ax_right.set_yticks([])
ax_right.set_xlabel("Total Exposure")

ax_heatmap = plt.subplot(gs[1, 0])
cbar_ax = plt.subplot(gs[1, 2])
sns.heatmap(similarity, cmap="viridis", annot=False, cbar=True, cbar_ax=cbar_ax, ax=ax_heatmap)
ax_heatmap.set_ylabel("SNMF Signatures")
ax_heatmap.set_xlabel("COSMIC Signatures")
ax_heatmap.tick_params(axis="x", rotation=90)

plt.tight_layout()

save_path = os.path.join(fig_dir, f"snmf_vs_cosmic_similarity_{adata_label}.pdf")
plt.savefig(save_path, bbox_inches="tight")
print(f"Saved: {save_path}")

plt.show()

print("\nHighest 20 COSMIC signatures by total exposure:")
print(cosmic_sums.sort_values(ascending=False).head(20))


In [ ]:
import os
import pandas as pd

# --- Define roots ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
output_root = os.path.join(project_root, "results", "reproduce", "TCGA")

# --- Path to de-novo NMF exposures ---
dn_exposures_path = os.path.join(
    output_root,
    "SBS96", "All_Solutions",
    "SBS96_21_Signatures",
    "Activities",
    "SBS96_S21_NMF_Activities.txt"
)

# --- Load de-novo exposures ---
dn_exposures = pd.read_csv(dn_exposures_path, sep="\t", index_col=0)
print(dn_exposures.shape)
dn_exposures


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- Define roots ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
output_root = os.path.join(project_root, "results", "reproduce", "TCGA")

# --- Load de-novo exposures (11 signatures) ---
dn_exposures_path = os.path.join(
    output_root,
    "count", "SBS96", "All_Solutions",
    "SBS96_11_Signatures",
    "Activities",
    "SBS96_S11_NMF_Activities.txt"
)
dn_exposures = pd.read_csv(dn_exposures_path, sep="\t", index_col=0)

# --- Normalize so each sample sums to 1 ---
dn_exposures = dn_exposures.div(dn_exposures.sum(axis=1), axis=0)

# --- Load COSMIC exposures (from adata objects) ---
cosmic_exposures = pd.concat([
    pd.DataFrame(adata_hrd.uns["cosmic_exposures_norm"], index=adata_hrd.obs_names),
    pd.DataFrame(adata_mmrd.uns["cosmic_exposures_norm"], index=adata_mmrd.obs_names)
], axis=0)

# --- Remove prefixes from index ---
dn_exposures.index = [idx.split("_", 1)[1] if "_" in idx else idx for idx in dn_exposures.index]

# --- Align sample indices ---
common_idx = dn_exposures.index.intersection(cosmic_exposures.index)
dn_exposures = dn_exposures.loc[common_idx]
cosmic_exposures = cosmic_exposures.loc[common_idx]

# --- Cosine similarity (between de-novo and COSMIC signatures) ---
similarity = pd.DataFrame(
    cosine_similarity(dn_exposures.T, cosmic_exposures.T),
    index=dn_exposures.columns,
    columns=cosmic_exposures.columns
)

# --- Marginal sums ---
cosmic_sums = cosmic_exposures.sum(axis=0)
dn_sums = dn_exposures.sum(axis=0)

# --- Plot ---
fig = plt.figure(figsize=(25, 7))
gs = gridspec.GridSpec(
    2, 3, width_ratios=[20, 2, 1], height_ratios=[12, 8],
    wspace=0.05, hspace=0.05
)

# --- Top barplot (COSMIC sums) ---
ax_top = plt.subplot(gs[0, 0])
ax_top.bar(cosmic_sums.index, cosmic_sums.values, color="gray")
ax_top.set_xlim(-0.5, len(cosmic_sums) - 0.5)
ax_top.set_xticks([])
ax_top.set_ylabel("Total Exposure")
ax_top.tick_params(axis="x", bottom=False)
ax_top.set_title("De-novo vs COSMIC Exposure Similarity (Cosine)", pad=20)

# --- Right barplot (de-novo sums) ---
ax_right = plt.subplot(gs[1, 1])
ax_right.barh(range(len(dn_sums)), dn_sums.values, color="gray")
ax_right.set_ylim(len(dn_sums) - 0.5, -0.5)
ax_right.set_yticks([])
ax_right.set_xlabel("Total Exposure")
# --- Heatmap ---
ax_heatmap = plt.subplot(gs[1, 0])
cbar_ax = plt.subplot(gs[1, 2])  # colorbar on far right
sns.heatmap(
    similarity, cmap="viridis", annot=False, 
    cbar=True, cbar_ax=cbar_ax, ax=ax_heatmap
)

# --- Add red squares for max similarity per SNMF signature ---
for i, row in enumerate(similarity.index):
    j = similarity.loc[row].idxmax()
    col_idx = similarity.columns.get_loc(j)
    rect = plt.Rectangle(
        (col_idx, i), 1, 1, fill=False, edgecolor="red", lw=2
    )
    ax_heatmap.add_patch(rect)

ax_heatmap.set_ylabel("SNMF Signatures")
ax_heatmap.set_xlabel("COSMIC Signatures")
ax_heatmap.tick_params(axis="x", rotation=90)


# --- Save figure (supplementary TCGA folder) ---
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "denovo_vs_cosmic_similarity_heatmap.pdf")
plt.savefig(fig_path, format="pdf", bbox_inches="tight", dpi=300)
print(f"Saved supplementary figure to {fig_path}")


## Exposure Distribution SNMF

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Prepare dataframes ---
def make_exposure_df(adata, label_col=None, label_value=None):
    df = pd.DataFrame(adata.uns['exposures_test'], index=adata.obs_names)
    df['cancer_type'] = (
        adata.obs[label_col].values if label_col in adata.obs.columns else [label_value] * adata.n_obs
    )
    return df

# HRD: all BRCA
df_hrd = make_exposure_df(adata_hrd, label_col='cancer_type', label_value='BRCA')

# MMRD: use adata.obs['cancer_type']
df_mmrd = make_exposure_df(adata_mmrd, label_col='cancer_type')

# Combine both
df_all = pd.concat([df_hrd, df_mmrd], axis=0)

# Melt to long format for seaborn
df_melt = df_all.melt(id_vars='cancer_type', var_name='Signature', value_name='Exposure')

# # Ensure consistent order
signature_order = [
    "SBS96A_HR", "SBS96B_MMR", "SBS96C_Control", "SBS96D_BER_UNG", "SBS96E_BER_OGG1"
]

cancer_order = ["BRCA", "COAD", "UCEC", "STAD"]

# --- Plot ---
sns.set(style="whitegrid", font_scale=1.1)
plt.figure(figsize=(10, 5))

palette = {
    "BRCA": "#4C78A8",  # HRD (blue)
    "COAD": "#F58518",  # colon
    "UCEC": "#54A24B",  # stomach
    "STAD": "#E45756"   # uterine
}

# Boxplot
ax = sns.boxplot(
    data=df_melt,
    x='Signature',
    y='Exposure',
    hue='cancer_type',
    order=signature_order,
    hue_order=cancer_order,
    palette=palette,
    width=0.6,
    fliersize=1.5,
    linewidth=1.2,
    boxprops=dict(alpha=0.9)
)

# Stripplot (aligned + transparent)
# Compute custom dodge offset to perfectly align dots with boxes
n_hues = len(cancer_order)
box_width = 0.6
dodge_width = box_width / n_hues

sns.stripplot(
    data=df_melt,
    x='Signature',
    y='Exposure',
    hue='cancer_type',
    order=signature_order,
    hue_order=cancer_order,
    dodge=True,
    size=3.5,
    alpha=0.2,  # more transparent
    linewidth=0,
    palette=palette
)

# Remove duplicate legends from stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:4], labels[:4], title="Cancer type", bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xlabel("SNMF Signatures")
ax.set_ylabel("Exposure")
ax.set_title("Exposure distribution across cancer types (test set)")
plt.xticks(rotation=15)
plt.tight_layout()

# --- Save figure ---
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
fig_dir = os.path.join(project_root, "results", "figures", "sup", "tcga")
os.makedirs(fig_dir, exist_ok=True)
save_path = os.path.join(fig_dir, "exposures_per_cancer.pdf")
plt.savefig(save_path, bbox_inches="tight")
print(f"Saved boxplot to {save_path}")

plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Define pathways and genes ---
pathway_genes = {
    "Control": ["ATP2B4"],
    "HR": ["EXO1", "RNF168"],
    "BER": ["UNG", "OGG1"],
    "MMR": ["MLH1", "MSH2", "MSH6", "PMS1", "PMS2"]
}

# --- Load SNMF exposures (λc=0) ---
bootstrap = "multinomial"
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
model_path = os.path.join(project_root, "results", "analysis", "bootstrap_comparison",
                          "SNMF_results_multinomial_lc0", "run_0")
exposures_path = os.path.join(model_path, "Exposures_test.txt")

exposures = pd.read_csv(exposures_path, sep="\t", index_col=0)
exposures.index = ['-'.join(s.split('-')[:3]) for s in exposures.index]  # simplify sample IDs

# --- Extract gene KO ---
exposures["gene_KO"] = [idx.split("_")[0] for idx in exposures.index]

# --- Average exposures per gene ---
avg_exposure_per_gene = exposures.groupby("gene_KO").mean().drop(columns=["gene_KO"], errors="ignore")

# --- Transpose: signatures × genes ---
signature_exposures = avg_exposure_per_gene.T

# --- Map genes → pathway ---
gene_to_pathway = {gene: pathway for pathway, genes in pathway_genes.items() for gene in genes}

# --- Average per pathway (signatures × pathways) ---
pathway_df = pd.DataFrame({
    pathway: signature_exposures[[g for g in genes if g in signature_exposures.columns]].mean(axis=1)
    for pathway, genes in pathway_genes.items()
})

# --- Dominant pathway per signature ---
dominant_pathway = pathway_df.idxmax(axis=1)
pathway_counts = dominant_pathway.value_counts()

# --- Top gene within each signature’s dominant pathway ---
top_gene_in_pathway = []
for sig in signature_exposures.index:
    pathway = dominant_pathway.get(sig, "Unknown")
    genes = pathway_genes.get(pathway, [])
    available = [g for g in genes if g in signature_exposures.columns]
    if available:
        top_gene = signature_exposures.loc[sig, available].idxmax()
    else:
        top_gene = "Unknown"
    top_gene_in_pathway.append(top_gene)

# --- Create annotated signature names ---
renamed_signatures = []
for sig, pw, gene in zip(signature_exposures.index, dominant_pathway, top_gene_in_pathway):
    if pw in pathway_counts and pathway_counts[pw] > 1 and gene != "Unknown":
        renamed_signatures.append(f"{sig}_{pw}_{gene}")
    else:
        renamed_signatures.append(f"{sig}_{pw}")

# Apply new signature names
signature_exposures.index = renamed_signatures

# --- Reorder genes by pathway for plot ---
ordered_genes = [gene for genes in pathway_genes.values() for gene in genes if gene in signature_exposures.columns]
signature_exposures = signature_exposures[ordered_genes]

# --- Save annotated exposures ---
annotated_output_path = os.path.join(model_path, "Avg_Signature_Exposure_Annotated.txt")
os.makedirs(os.path.dirname(annotated_output_path), exist_ok=True)
signature_exposures.to_csv(annotated_output_path, sep="\t")
print(f"✅ Saved annotated exposures to:\n{annotated_output_path}")

# --- Plot heatmap ---
plt.figure(figsize=(8, 5))
ax = sns.heatmap(
    signature_exposures,
    cmap="viridis",
    linewidths=0.3,
    linecolor="lightgray",
    cbar_kws={"label": "Avg Signature Exposure"}
)
plt.title("Average Signature Exposure per Gene KO (λc=0)")
plt.xlabel("Gene KO")
plt.ylabel("Annotated Signature")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

# Add separators between pathways
separators = np.cumsum([len(v) for v in pathway_genes.values()])[:-1]
for sep in separators:
    ax.axvline(sep, color="white", linewidth=3)

plt.tight_layout()
plt.show()
